[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/ICA_Blind_Source_Separation.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Blind Source Separation & ICA

The cocktail-party problem, actually solved: several microphones each hear a *mixture* of sources, and — knowing nothing about the mixing — we unmix them. The key is a beautiful statistical loophole: Gaussianity is the one thing mixing *increases*. Verified the only way that matters: recovered sources correlate ≈1 with the planted truth.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3, [Independence](../Intro_Math/Analysis/Independence.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

# three planted sources: a chirp 'voice', a square-wave 'hum', an impulsive 'percussion'
fs, T_dur = 8000, 3.0
t = np.arange(0, T_dur, 1/fs)
s1 = sig.chirp(t, 300, T_dur, 800) * (1 + 0.3*np.sin(2*np.pi*2*t))
s2 = sig.square(2*np.pi*120*t) * 0.7
s3 = np.zeros_like(t)
for tc in rng.uniform(0, T_dur, 25):
    i = int(tc*fs); s3[i:i+150] += np.exp(-np.arange(150)/25) * rng.choice([-2, 2])
S_true = np.stack([s1, s2, s3])
S_true = (S_true - S_true.mean(1, keepdims=True)) / S_true.std(1, keepdims=True)

A_mix = rng.standard_normal((3, 3))                    # unknown room acoustics
X = A_mix @ S_true                                      # what the microphones record

---
### 🕐 Session 1 of 3 — *The Problem & Why Correlation Isn't Enough* (~35 min)
**Goal:** see mixing destroy the sources; understand why PCA/whitening only gets you halfway.
**Feeds into:** Session 2 (the non-Gaussian loophole).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Problem & Why Correlation Isn't Enough</b></summary>

**Timing (~35 min).** 8 min the cocktail-party setup · 10 min whitening · 10 min the rotation-blindness argument · 7 min buffer.

**Open with the problem stated as impossible.** Three microphones, three unknown sources, an unknown mixing matrix. We know neither $A$ nor $S$ and want both from $X = AS$ alone. Ask the room whether that is even well-posed — for any invertible $R$, $X = (AR)(R^{-1}S)$ fits equally well, so there are infinitely many factorisations. The workshop's job is to explain what extra assumption makes one of them correct, and *independence* turns out to be enough. Framing it as an ill-posed problem that a single assumption rescues is much better than presenting ICA as an algorithm.

**Whitening is the half-solution — make its limit vivid.** Decorrelating via the covariance eigendecomposition removes all second-order structure, and it does undo a lot of the mixing. Then the key argument: if $Z$ has covariance $I$, so does $QZ$ for *any* orthogonal $Q$, since $QIQ^\top = I$. Second-order statistics are therefore **rotation-blind**. Draw it: whitening takes you to a sphere of equally-valid candidate unmixings and then falls silent. Something beyond variance must choose the point on that sphere.

**Ask the room.** "Covariance has been exhausted. What information is left in the data?" The answer — everything above second order, i.e. the *shape* of the distributions rather than their spread — sets up Session 2 exactly. If [Cyclostationary & HOS](./Cyclostationary_HOS.ipynb) is on the syllabus, this is the same "second order is not everything" theme arriving from a different direction.

**Point at the numbers rather than the plots.** The mixtures look like noise, which is convincing but qualitative. The quantitative statement is the printed correlation of $Z_1$ with the three true sources: 0.72, 0.63, 0.28. Whitening left the first whitened channel correlated with *two* sources at once. That is the residual rotation, measured.

**Note the deliberate source design.** A chirp, a square wave, and sparse impulsive bursts — chosen to be strongly non-Gaussian in different ways, since that is precisely the resource Session 2 spends. Say so now, because Session 3's Gaussian failure is only surprising if students know the success was engineered.
</details>

## 2. Three Microphones, Three Tangles

💡 **Intuition.** Each mic hears $x_i = \sum_j a_{ij} s_j$: linear, instantaneous mixing. **Whitening** (decorrelating via the covariance [eigendecomposition](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)) can undo mixing *up to a rotation* — but second-order statistics are **rotation-blind**: every rotation of white signals is equally white. Correlation has taken you to a sphere of candidate unmixings and gone silent. Something beyond variance must pick the rotation — that something is Session 2.

In [2]:
fig, axes = plt.subplots(2, 3, figsize=(10.5, 3.2))
for ax, s, name in zip(axes[0], S_true, ["'voice' (chirp)", "'hum' (square)", "'percussion'"]):
    ax.plot(t[:2000], s[:2000], linewidth=0.7); ax.set_title("source: " + name, fontsize=8)
for ax, x in zip(axes[1], X):
    ax.plot(t[:2000], x[:2000], linewidth=0.7, color="crimson"); ax.set_title("a microphone", fontsize=8)
plt.tight_layout(); plt.show()

# whiten
Xc = X - X.mean(1, keepdims=True)
C = Xc @ Xc.T / Xc.shape[1]
w_eig, V_eig = np.linalg.eigh(C)
W_white = np.diag(w_eig**-0.5) @ V_eig.T
Z = W_white @ Xc
print("after whitening, covariance = I:", np.allclose(Z @ Z.T / Z.shape[1], np.eye(3), atol=1e-10))
print("...but the sources are still mixed: |corr| of Z1 with each true source:",
      np.abs([np.corrcoef(Z[0], s)[0,1] for s in S_true]).round(2))

after whitening, covariance = I: True
...but the sources are still mixed: |corr| of Z1 with each true source: [0.72 0.63 0.28]


/tmp/ipykernel_2985363/4044920668.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The top row shows three structured sources — a chirp, a square wave, sparse impulsive bursts — and the bottom row shows what the microphones actually record: three tangles in which none of that structure is visible. Mixing destroyed the signals as far as the eye is concerned.

Then whitening runs, and the printed check confirms it worked in its own terms: the covariance of $Z$ is the identity to within 1e-10. All second-order structure has been removed. But the next line is the important one — the correlation of the first whitened channel with the three true sources is **0.72, 0.63, 0.28**. Channel 1 is still a blend of at least two sources. Whitening decorrelated the data without unmixing it.

**Why it necessarily stops there.** If $Z$ has covariance $I$, then so does $QZ$ for *any* orthogonal matrix $Q$, because $Q I Q^\top = I$. Whitening therefore determines the solution only **up to a rotation** — it has narrowed infinitely many candidate unmixings down to a sphere of them, and every point on that sphere is exactly as white as every other. Second-order statistics are *rotation-blind*, and no amount of cleverness with covariance will break the tie. The 0.72/0.63 pair is that unresolved rotation, measured.

This is worth stating as a general limit rather than a quirk of this dataset. PCA and whitening exhaust everything covariance knows; if the structure you need survives in the residual rotation, you must look beyond second order to find it.

**Which is also why the problem looked ill-posed to begin with.** For any invertible $R$, $X = (AR)(R^{-1}S)$ is a perfectly good factorisation, so nothing in $X$ alone singles out the true $A$ and $S$. Some additional assumption must do that work. Session 2 supplies it — and remarkably, the assumption that the sources are *statistically independent and non-Gaussian* turns out to be sufficient.

One design note to keep in mind: the three sources here were chosen to be strongly non-Gaussian in different ways — a chirp, a square wave, and sparse bursts. That is not incidental. It is exactly the resource Session 2 will spend, and Session 3 shows what happens when it is unavailable.

---
### 🕐 Session 2 of 3 — *The Non-Gaussian Loophole & FastICA* (~40 min)
**Goal:** the CLT in reverse: mixtures are MORE Gaussian than sources — so maximize non-Gaussianity.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (limits & practice).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: The Non-Gaussian Loophole & FastICA</b></summary>

**Timing (~40 min).** 12 min the CLT-in-reverse argument · 10 min FastICA mechanics · 12 min the demo and its oracle · 6 min the built-in ambiguities.

**The CLT run backwards is the idea of the whole workshop — give it the time.** The [Central Limit Theorem](../Intro_Math/Analysis/Independence.ipynb) says sums of independent things drift toward Gaussian. Each microphone is a sum of sources. Therefore **every mixture is more Gaussian than any source in it.** Turn that into a search: rotate the whitened data until each output is as *non-Gaussian as possible*, and you must be un-mixing, because mixing can only move you the other way. That is all of ICA, and students should be able to state it in one sentence by the end of the session.

**Ask the room before proceeding.** "Why does maximising non-Gaussianity find *sources* rather than something arbitrary?" Because Gaussianity is monotone under mixing — it is the one quantity that only ever increases. The compass points toward unmixing precisely because there is no way to become less Gaussian by mixing more. This also explains why Session 3's failure case is a failure *of the compass*, not of the algorithm.

**Do not over-explain the fixed-point iteration.** The substance is: $\log\cosh$ is a smooth, robust surrogate for non-Gaussianity (its derivative is $\tanh$, which appears in the code), and the deflation loop is Gram–Schmidt, forcing each new direction orthogonal to the ones already found. Note why orthogonality is legitimate here: after whitening, independent directions *are* orthogonal, so the constraint costs nothing. Students often assume deflation is an approximation; it is not, given whitening.

**Name the ambiguities before someone objects.** Order, sign, and scale are unrecoverable — permuting sources or negating one produces an equally valid solution, since neither operation changes independence or non-Gaussianity. That is why the demo compares with `np.abs` on correlations and why the plotting cell computes a `flip`. This is not sloppiness; it is a genuine property of the problem.

**Frame the oracle properly — it is a strong test.** Correlating recovered against true sources gives a $3\times3$ matrix, and the requirement is not merely that the diagonal is large. It is that each recovered source matches **exactly one** true source (near-1 in one entry, near-0 elsewhere) and that the argmax is a *permutation* — which is what `len(set(corr.argmax(1))) == 3` checks. A degenerate result where two outputs both matched source 1 would pass a naive "correlations are high" test and fail this one.

**Remind the room what was not used.** `A_mix` is never referenced after generating `X`. No knowledge of the room's acoustics, the number of sources' characteristics, or their spectra entered the algorithm — only the assumption that they are independent and non-Gaussian. That is what "blind" means, and it is worth saying explicitly at the moment the result appears.
</details>

## 3. Gaussianity as a Compass

💡 **Intuition.** The [CLT](../Intro_Math/Analysis/Independence.ipynb) says sums of independent things drift *toward* Gaussian. Flip it around: each microphone (a sum of sources) is **more Gaussian than any single source** — so to unmix, rotate the whitened data until each output is as **non-Gaussian as possible**. That's all of ICA. FastICA does it with a fixed-point iteration on a smooth non-Gaussianity score (we use $\log\cosh$), one source at a time, deflating (Gram–Schmidt) so each new direction is orthogonal to the found ones. Built-in limits fall out of the logic: source order and sign/scale are unrecoverable, and **two Gaussian sources cannot be separated** (their mixtures are exactly rotation-symmetric).

In [3]:
def fastica(Z, n_comp, iters=300):
    W = np.zeros((n_comp, Z.shape[0]))
    for k in range(n_comp):
        w = rng.standard_normal(Z.shape[0]); w /= np.linalg.norm(w)
        for _ in range(iters):
            u = w @ Z
            g, gp = np.tanh(u), 1 - np.tanh(u)**2            # logcosh score & derivative
            w_new = (Z * g).mean(1) - gp.mean() * w
            for j in range(k):                                # deflate: stay ⊥ to found sources
                w_new -= (w_new @ W[j]) * W[j]
            w_new /= np.linalg.norm(w_new)
            if np.abs(np.abs(w_new @ w) - 1) < 1e-10: w = w_new; break
            w = w_new
        W[k] = w
    return W

W_ica = fastica(Z, 3)
S_hat = W_ica @ Z

# ORACLE: each recovered source must match ONE true source with |corr| ≈ 1
corr = np.abs(np.corrcoef(np.vstack([S_hat, S_true]))[:3, 3:])
print("correlation matrix (recovered × true):\n", corr.round(3))
best = corr.max(1)
print(f"per-source best |corr|: {best.round(4)}  — separation achieved: {bool((best > 0.98).all())}")
assert (best > 0.98).all() and len(set(corr.argmax(1))) == 3

correlation matrix (recovered × true):
 [[1.    0.015 0.   ]
 [0.003 0.035 1.   ]
 [0.008 0.999 0.008]]
per-source best |corr|: [1.     1.     0.9993]  — separation achieved: True


**What just happened.** Per-source best correlations of **[1.000, 1.000, 0.9993]** against the planted truth — the cocktail party, solved. And the correlation *matrix* is the real evidence, not those three numbers:

```
[[1.    0.015 0.   ]
 [0.003 0.035 1.   ]
 [0.008 0.999 0.008]]
```

Each row has exactly one near-1 entry and near-zeros elsewhere, and the argmax positions form a **permutation** — recovered source 1 ↔ true source 1, 2 ↔ 3, 3 ↔ 2. That is a much stronger claim than "the correlations are high." A degenerate solution in which two outputs both latched onto the loudest source would still show some large correlations; it would fail the permutation check, which is exactly why the `assert` tests `len(set(corr.argmax(1))) == 3`. Off-diagonal entries at the 0.01 level mean each output contains essentially none of the other sources.

**What the algorithm was not given.** `A_mix` is never referenced after `X` is constructed. No knowledge of the mixing, the room, the number of microphones relative to sources, the sources' spectra, or their timing entered the computation. The only assumption was that the sources are statistically independent and non-Gaussian. That is what makes this *blind* separation, and it is worth pausing on: an ill-posed factorisation became uniquely solvable on the strength of one statistical assumption.

**Why the assumption is enough.** The [CLT](../Intro_Math/Analysis/Independence.ipynb) says sums of independent quantities drift toward Gaussian, so each microphone — being a sum of sources — is *more Gaussian than any source in it*. Gaussianity is monotone under mixing: there is no way to become less Gaussian by mixing more. So "rotate until each output is maximally non-Gaussian" is a compass that can only point toward unmixing. FastICA follows it with a fixed-point iteration on $\log\cosh$ (hence the `tanh` in the code), deflating by Gram–Schmidt so each new direction is orthogonal to those already found — legitimate because after whitening, independent directions *are* orthogonal.

**Two things ICA structurally cannot recover, visible in the output above.** The **order** is permuted rather than preserved, and the **sign and scale** are arbitrary — which is why the comparison uses `np.abs` and why the plot below computes a `flip`. Neither permuting sources nor negating one changes their independence or their non-Gaussianity, so nothing in the problem statement distinguishes those solutions. These are genuine properties of blind separation, not defects of the implementation, and any downstream use has to tolerate them.

**And a caveat on how favourable this is.** The three sources were chosen to be strongly non-Gaussian in different ways, the mixing is instantaneous (a plain matrix, with no delays), and there are exactly as many microphones as sources. Session 3 removes the first of those conditions and watches the method fail.

In [4]:
fig, axes = plt.subplots(3, 1, figsize=(9, 3.6), sharex=True)
order = corr.argmax(1)
for ax, sh, idx in zip(axes, S_hat, order):
    flip = np.sign(np.corrcoef(sh, S_true[idx])[0, 1])
    ax.plot(t[:2000], flip*sh[:2000], linewidth=0.7, label="recovered")
    ax.plot(t[:2000], S_true[idx][:2000], "k--", linewidth=0.5, alpha=0.6, label="truth")
    ax.legend(fontsize=6, loc="upper right")
plt.suptitle("unmixed blind — no knowledge of the mixing matrix was used")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2985363/187698282.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The recovered traces sit on top of the dashed truth almost everywhere — the chirp's sweep, the square wave's hard edges, and the sparse bursts all reconstructed from three microphone signals that individually looked like noise.

The waveform view adds something the correlation numbers cannot: it shows *where* the reconstruction is good. Correlation is a single summary, and a value of 0.999 could in principle hide a badly-reconstructed transient. Here the square wave's discontinuities and the percussion's sharp onsets are reproduced cleanly, which matters because those are precisely the features a separation method that had merely captured the dominant subspace would smear.

**Note the two corrections the plot has to make.** `order = corr.argmax(1)` re-sorts the outputs to match the truth, and `flip = np.sign(...)` fixes each trace's sign. Both are necessary because ICA cannot recover permutation or sign — negating a source or reordering the set changes neither its independence nor its non-Gaussianity, so nothing distinguishes those solutions. Being explicit about this is the honest way to plot: the alignment is applied for *display*, and no information from the truth was used during separation.

That distinction is worth insisting on, because it is a place where a demo can quietly cheat. The unmixing matrix came from `fastica(Z, 3)`, which sees only the whitened microphone data. The true sources appear afterwards, twice: once to compute the correlation oracle, and once to order and flip the traces for this figure. Neither influenced the result.

**What this looks like in the field.** EEG artifact removal runs exactly this pipeline — eye blinks are gloriously non-Gaussian, so ICA isolates them into their own component, which is then simply deleted before reconstructing the clean signal. Same three lines, same assumptions.

Session 3 now removes the assumption that made all of this work.

---
### 🕐 Session 3 of 3 — *Limits, Diagnostics & Practice* (~30 min)
**Goal:** what ICA can't do, how to sanity-check it, and where it runs in the wild.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Limits, Diagnostics & Practice</b></summary>

**Timing (~30 min).** 10 min the Gaussian impossibility · 8 min the demo and how to read it · 12 min the field guide.

**Frame the Gaussian case as a theorem, not a weakness.** Two Gaussian sources are *provably* unseparable, and the reason is clean enough to give in full: a rotation of independent Gaussians is again independent Gaussians with the same distribution. The mixture is exactly rotation-symmetric, so there is no information anywhere in the data — at any statistical order — that identifies which rotation is "correct." The compass does not merely point weakly; it points nowhere, because every direction is genuinely equivalent. Say plainly that no algorithm can beat this, ever. It is a limit of the *problem*, not of FastICA.

**Ask the room to predict the output before running.** Most expect low correlations, around 0.5. The actual result is `[0.784, 0.779, 0.76, 0.968, 0.972]` — often quite *high*, and different every restart. Ask what is going on. With only two sources, a randomly chosen rotation lands near a true source reasonably often by luck, so any single run can look like a success. **The tell is not the level, it is the scatter.** This is a genuinely important lesson: a method that gives a different answer each time it is run has not solved anything, however good any individual answer looks.

**Turn that into the practical diagnostic.** Restart consistency is the field's identifiability certificate. Run ICA several times from different random initialisations: consistent answers mean the rotation was determined by the data; scattered answers mean it was not. Combine with a kurtosis check on the outputs — components that are near-Gaussian are the ones you cannot trust. Both take three lines and both should be habitual.

**Work the field guide as scenarios, not a list.** Pose situations and let the room classify them. *EEG with eye blinks* — works, blinks are extremely non-Gaussian and get isolated into a component you then delete. *Two speakers in a reverberant room* — fails as written, because real rooms delay as well as scale, making the mixing convolutive rather than instantaneous; the fix is frequency-domain ICA, one instantaneous problem per frequency bin, with a permutation-alignment problem across bins. *Five sources, two microphones* — underdetermined, so no invertible unmixing exists at all; you need [sparsity](./Sparse_Dictionary_Learning.ipynb) instead.

**Close on the lineage if you have time.** Nonlinear ICA is provably impossible without auxiliary structure, which is a live research frontier and the reason modern disentanglement work in [representation learning](../Intro_Mach_Learn/Representation_Learning.ipynb) leans on temporal structure or auxiliary variables. Students who found the CLT-in-reverse argument satisfying will find it satisfying that the field knows exactly where the argument stops working.
</details>

## 4. The Honest Fine Print

In [5]:
# the promised failure: two GAUSSIAN sources are unseparable — watch it happen
S_gauss = rng.standard_normal((2, 40000))
X_g = rng.standard_normal((2, 2)) @ S_gauss
Cg = X_g @ X_g.T / X_g.shape[1]
wg, Vg = np.linalg.eigh(Cg)
Zg = np.diag(wg**-0.5) @ Vg.T @ X_g
best_corrs = []
for trial in range(5):                                # multiple restarts — none will succeed
    Wg = fastica(Zg, 2, iters=200)
    cg = np.abs(np.corrcoef(np.vstack([Wg @ Zg, S_gauss]))[:2, 2:])
    best_corrs.append(cg.max(1).min())
print(f"Gaussian sources, worst-recovered |corr| across 5 restarts: {np.round(best_corrs, 3)}")
print("→ compare the non-Gaussian case: 0.999+ EVERY time. Here the answers scatter with the")
print("  random start — the rotation is unidentifiable, and restart-inconsistency is the tell.")

Gaussian sources, worst-recovered |corr| across 5 restarts: [0.784 0.779 0.76  0.968 0.972]
→ compare the non-Gaussian case: 0.999+ EVERY time. Here the answers scatter with the
  random start — the rotation is unidentifiable, and restart-inconsistency is the tell.


**Field guide.**

- **Works:** EEG artifact removal (eye blinks are gloriously non-Gaussian), [audio](./Audio_Speech_DSP.ipynb) unmixing with instantaneous mixtures, hyperspectral unmixing.
- **Fails or needs upgrades:** convolutive/reverberant mixing (rooms delay, not just scale — needs frequency-domain ICA), more sources than mics (underdetermined → [sparsity](./Sparse_Dictionary_Learning.ipynb) to the rescue), Gaussian-ish sources.
- **Diagnostics:** always check kurtosis of outputs (should be far from 0), and run restarts — consistent answers across restarts are the practical identifiability certificate.
- **Lineage:** [contrastive learning](../Intro_Mach_Learn/Representation_Learning.ipynb) and modern disentanglement research are ICA's descendants (nonlinear ICA is provably impossible without auxiliary structure — a live research frontier).

## 5. Conclusion

Whitening gets you to a rotation; the CLT-in-reverse picks it; FastICA computes it (recovered × truth correlations > 0.99, verified); and Gaussian sources mark the hard boundary of the possible (also verified). Blindness, it turns out, is negotiable — Gaussianity isn't.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — unmixing with *geometry* instead of statistics.
- [Representation Learning](../Intro_Mach_Learn/Representation_Learning.ipynb) — the neural descendants.
- [Manifold Optimization](../Intro_Math/Optimization/Manifold_Optimization.ipynb) — ICA's rotation search lives on the Stiefel manifold.